# Use case — `Utility/time_delay_interpolation.py`

Classical two-curve baseline using three linear-interpolation comparisons and their symmetric mean.

**Convention:** `t_B_shifted = t_B - delay`. A positive delay means B is observed later than A. Start with the `quick` profile or the explicit parameters below, then inspect every score profile before running Monte Carlo uncertainty.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility import time_delay_interpolation as td

## Input identifiers

Use strings for Gaia IDs to prevent accidental floating-point rounding.

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"  # preprocessed light curves.
SOURCE_ID = "Source_ID"                               # System containing both light curves.
COMPONENT_A = "Component_A_ID"                             # Reference component; its delay is fixed to zero.
COMPONENT_B = "Component_B_ID"                             # Component whose delay relative to A is estimated.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")

## Estimator hyperparameters

The dictionary below reproduces the original notebook adaptation. Equivalent named profiles are available in `configs/time_delay_profiles.json`.

In [ ]:
estimator_kwargs = {
    "dmin": -500,              # Smallest tested B-minus-A delay, in days.
    "dmax": 500,               # Largest tested B-minus-A delay, in days.
    "ngrid": 400,              # Number of coarse delay-grid values between dmin and dmax.
    "common_grid_size": 200,   # Points used only for interpolated-A versus interpolated-B.
    "fit_offset": True,        # Fit one constant vertical flux offset for each comparison.
    "sigma_floor": 0.05,       # Minimum normalised measurement error; prevents extreme weights.
                               # maximum(flux_error / sigma, sigma_floor).
    "min_points": 10,          # Minimum raw overlap points required in each component.
    "min_frac": 0.50,          # Minimum fraction of each component retained in overlap.
    "min_span_frac": 0.50,     # Minimum fraction of the original temporal span in overlap.
    "overlap_penalty": 0.0,    # Extra cost for lost span/fraction; zero keeps hard constraints only.
                               # Goal is reduce bias from short overlap intervals, but this can also biais the true result.
    "refine": True,            # Refine the best grid value with bounded scalar optimisation.
    "scalar_xatol": 0.05,      # Local optimiser tolerance in days.
                               # Tolerance of the convergence of the local optimiser; smaller values are more accurate but slower.
    "verbose": True,           # Print the four method estimates and diagnostics.
}

MC_SAMPLES = 300              # Independent flux-error draws; use 20 for a test and >=300 for final work.
MC_RANDOM_SEED = 42           # Makes uncertainty samples reproducible.
MC_ERROR_SCALE = 1.0          # 1.0 uses flux_obs_error exactly; 2.0 doubles every measurement sigma.
MC_PROGRESS_EVERY = 25        # Print progress every N draws; set 0 to suppress progress lines.

In [ ]:
df = td.load_lightcurve_csv(INPUT_CSV)
system = td.get_pair_from_df(
    df,
    source_id=SOURCE_ID,
    comp_a=COMPONENT_A,
    comp_b=COMPONENT_B,
    names=("A reference", "B"),
)

result = td.estimate_time_delay_linear_interpolation(
    system["curves"],
    **estimator_kwargs,
)
td.plot_delay_profiles_interpolation(result)
td.plot_best_alignments_interpolation(result)

## Measurement-error uncertainty

This is Monte Carlo propagation of `flux_obs_error`, not MCMC sampling of a delay posterior. Inspect histograms for multiple modes.

In [ ]:
uncertainty = td.run_fluxobs_error_mc_interpolation(
    system=system,
    estimator_kwargs=estimator_kwargs,
    n_samples=MC_SAMPLES,
    random_seed=MC_RANDOM_SEED,
    error_scale=MC_ERROR_SCALE,
    base_res=result,
    progress_every=MC_PROGRESS_EVERY,
    verbose=True,
)
td.print_fluxobs_error_mc_interpolation(uncertainty)
td.plot_fluxobs_error_mc_interpolation(uncertainty, bins=30)
display(uncertainty["summary"])

## Command-line equivalent

```bash
python -m Utility.time_delay_interpolation data/cleaned_lightcurves.csv --source-id 3361094865862486656 --component-a 3361094865862486721 --component-b 3361094865862486723 --profile legacy_notebook --mc-samples 300
```